# M1 and M2: PyTorch on the Mac, then the data loader

How to use this notebook (see `docs/TRANSFORMER_GUIDE.md` for the why):
- Pick the kernel **Hoopformer (.venv)** (top right). Run the Setup cell first.
- Write your code in the cells marked **✍️ YOUR CODE**.
- Then run the **✅ CHECK** cell below it. Checks print what they look at and fail loudly when something is off.
- When M2's checks all pass, move your code into `src/hoopformer/data.py` with tests (last section).

In [ ]:
# Setup: run this first. It moves to the project folder so paths like data/... work,
# and reloads project code automatically whenever you edit src/hoopformer/.
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("working in", Path.cwd())

## M1: PyTorch and the Mac's GPU

Learn: tensors, devices, and why the GPU wins on big matrix products. On Apple silicon the gap is smaller than you'd expect, because the CPU has built-in matrix units.

### 1a. Can PyTorch see the GPU?
✍️ `import torch`, print `torch.__version__`, and set `device = "mps"` if `torch.backends.mps.is_available()` else `"cpu"`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 1a
print("torch", torch.__version__, "| device:", device)
assert device == "mps", "PyTorch can't see the GPU: run `uv sync` and make sure this notebook uses the Hoopformer (.venv) kernel"

### 1b. Tensors and devices
✍️ Make `a`, a 3×4 tensor of random numbers on `device`, and `b`, a 4×2 one (`torch.randn(..., device=device)`). Compute `c = a @ b`. **Predict `c`'s shape before you run it.** Then copy it to the CPU: `c_cpu = c.cpu()`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 1b
print("c:", tuple(c.shape), c.device, "| c_cpu:", c_cpu.device)
assert tuple(c.shape) == (3, 2), "(3×4) @ (4×2) gives 3×2: the inner sizes must match and disappear"
assert c.device.type == device and c_cpu.device.type == "cpu"

### 1c. The race: GPU vs CPU
✍️ Write `time_matmul(device, n=4096, reps=10)`:
1. make two n×n random tensors on `device`
2. multiply them once to warm up
3. time `reps` multiplies with `time.perf_counter()`
4. return the seconds per multiply

**Trap:** GPU work is asynchronous. Python gets control back before the GPU has finished, so on `"mps"` call `torch.mps.synchronize()` before reading the clock (both after the warm-up and after the loop), or you'll be timing nothing.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 1c
cpu_s, gpu_s = time_matmul("cpu"), time_matmul("mps")
tflops = lambda seconds: 2 * 4096**3 / seconds / 1e12  # a multiply of two n×n matrices is 2n³ operations
print(f"CPU {cpu_s * 1000:.1f} ms = {tflops(cpu_s):.2f} TFLOPS | GPU {gpu_s * 1000:.1f} ms = {tflops(gpu_s):.2f} TFLOPS | GPU is {cpu_s / gpu_s:.1f}x faster")
print("reference, MacBook Air M5: CPU 1.96, GPU 2.74 TFLOPS (1.4x)")
assert tflops(gpu_s) < 200, "impossibly fast: a missing torch.mps.synchronize() means you timed nothing"
assert 0.2 < tflops(cpu_s) and 0.2 < tflops(gpu_s)

Write your two TFLOPS numbers and the ratio in `docs/EXPERIMENTS.md`: that's the first line of your lab notebook. **M1 is done.**

## M2: the data loader

First build the dataset once, in a terminal (about 1.5 minutes on the Studio):
```
uv run hoopformer dataset
```

### 2a. Become one with the data (Karpathy, step 1)
✍️ Load `data/derived/possessions.parquet` into `df` with pandas. Look at:
- `df.shape` and `df.head()`
- rows per `split`
- the share of each `points` value
- `df.start_margin.describe()`

Write down one thing that surprised you.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 2a
print(df.shape)
print(df.split.value_counts().to_dict())
assert set(df.split) == {"train", "validation", "test"}
assert list(df.columns[:3]) == ["game_id", "season", "split"]

### 2b. The vocabulary: player id → row of the embedding table
✍️ Write `build_vocab(train_df)` returning a dict `{personId: index}`. Reserve 0 = PAD, 1 = UNK (unknown player), 2 = MASK (used in M6), so real players start at 3.

**Use only the training rows.** Why? A vocabulary built from all splits would quietly tell the model which players exist in the future: leakage.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 2b
import pandas as pd
PLAYER_COLUMNS = [f"off_{i}" for i in range(5)] + [f"def_{i}" for i in range(5)]
train = df[df.split == "train"]
vocab = build_vocab(train)
players_in_train = set(pd.unique(train[PLAYER_COLUMNS].to_numpy().ravel()))
print(len(vocab), "players in the vocabulary")
assert set(vocab) == players_in_train, "every training player, and only training players"
assert sorted(vocab.values()) == list(range(3, 3 + len(vocab))), "indices 3, 4, 5, ... with no gaps"

### 2c. Who is unknown?
✍️ Set `unseen` to the set of players who appear in the **test** split but not in `vocab`. Print how many there are and a few of their ids. These are the players your model will see as UNK.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 2c
test_players = set(pd.unique(df[df.split == "test"][PLAYER_COLUMNS].to_numpy().ravel()))
print(len(unseen), "test-season players the model has never seen")
assert unseen == test_players - set(vocab)
assert len(unseen) > 0, "every season brings rookies"

### 2d. The Dataset
✍️ Write `class PossessionDataset(torch.utils.data.Dataset)` that takes `(df, vocab)`.
- `__len__` returns the number of rows.
- `__getitem__(i)` returns a dict with:
  - `"players"`: a LongTensor of 10 indices, offense then defense; unknown players → 1
  - `"situation"`: a FloatTensor `[period / 4, seconds_left / 720, clip(start_margin, -20, 20) / 20, float(offense_is_home)]`
  - `"label"`: a LongTensor holding `points`

**Hint:** turn the columns into numpy arrays once in `__init__`. Reading rows through pandas one at a time is thousands of times slower.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 2d
validation = PossessionDataset(df[df.split == "validation"], vocab)
item = validation[0]
print(len(validation), "rows |", {key: (tuple(value.shape), value.dtype) for key, value in item.items()})
assert item["players"].shape == (10,) and item["players"].dtype == torch.long
assert item["situation"].shape == (4,) and item["situation"].dtype == torch.float32
assert item["label"].dtype == torch.long and 0 <= int(item["label"]) <= 4
assert int(item["players"].min()) >= 1, "0 is PAD and must never appear in a real lineup"
test = PossessionDataset(df[df.split == "test"], vocab)
with_unknown = sum(bool((test[i]["players"] == 1).any()) for i in range(2000))
print(f"{with_unknown} of the first 2,000 test possessions include a player the model has never seen")
assert with_unknown > 0

### 2e. Batches
✍️ Wrap the validation dataset in `torch.utils.data.DataLoader(validation, batch_size=2048, shuffle=True)` and take one batch with `batch = next(iter(loader))`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 2e
print({key: tuple(value.shape) for key, value in batch.items()})
assert batch["players"].shape == (2048, 10) and batch["situation"].shape == (2048, 4) and batch["label"].shape == (2048,)

### 2f. Move it into the package
When every check passes:
1. Copy `build_vocab` and `PossessionDataset` into **`src/hoopformer/data.py`**.
2. Turn the checks above into tests in **`tests/test_data.py`**: real data, no mocks, print what each test sees.
3. `uv run pytest` must pass.
4. Commit with a message that says what and why, then push, and ask Claude to review.

**M2 is done.** Next is M3: your first model (a bag of players).